# Plaza Brawler: AI art on a free GPU (Kaggle or Colab)

Makes the fighter sprites, the Halo Plaza background, hit effects and UI icons listed in
`tools/asset_gen/assets.yaml`, with **FLUX.1 [schnell]** (Apache-2.0), then zips them for download.

**Before you press Run all**

1. **Kaggle:** right panel → *Session options* → **Accelerator: GPU T4 x2** (or GPU T4). P100 may not
   work with current PyTorch. Turn **Internet on** (needs a phone-verified Kaggle account).
   **Colab:** *Runtime → Change runtime type → T4 GPU*.
2. **Hugging Face token (needed):** FLUX.1 [schnell] is free and Apache-2.0, but Hugging Face makes you
   log in and click *Agree* on https://huggingface.co/black-forest-labs/FLUX.1-schnell once.
   Then make a **Read** token at https://huggingface.co/settings/tokens and add it as a secret named
   **`HF_TOKEN`**: Kaggle *Add-ons → Secrets*, Colab the 🔑 icon on the left. Never paste it into a cell.
3. Run all cells. The first run downloads about 34 GB of model files (a few minutes on Kaggle);
   each image then takes roughly 20–60 s on a T4. A full set is 22 images plus 2 sprite sheets.


In [ ]:
# Settings: change these if you like
BRANCH = "feat/ai-asset-pipeline"   # change to "main" after the pull request is merged
ONLY = []                           # e.g. ["fighters/aries/*"] for one fighter, [] for everything
STYLE = "clean2d"                   # or "pixel"
EXTRA_ARGS = []                     # e.g. ["--skip-existing"]

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())
os.environ["HF_HOME"] = "/tmp/hf"          # big scratch disk, not the 20 GB output folder
os.environ["U2NET_HOME"] = "/tmp/u2net"     # rembg background-removal weights
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other", "| work dir:", WORK)
subprocess.run(["nvidia-smi"], check=False)
import torch
assert torch.cuda.is_available(), "No GPU. Turn on the GPU accelerator (see the first cell) and restart."
major, minor = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), f"sm_{major}{minor}", f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB", "torch", torch.__version__)
if major < 7:
    print("WARNING: this GPU (e.g. P100) may not be supported by this PyTorch/bitsandbytes. Switch to GPU T4 if anything fails.")

In [ ]:
# Hugging Face login from the HF_TOKEN secret
token = None
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Kaggle secret HF_TOKEN not found:", e)
elif ON_COLAB:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("Colab secret HF_TOKEN not found:", e)
if not token:
    from getpass import getpass
    token = getpass("Hugging Face read token: ")
os.environ["HF_TOKEN"] = token
from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print("Logged in to Hugging Face.")

In [ ]:
# Get the code
REPO = os.path.join(WORK, "Plaza-Brawler")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/kiroba/Plaza-Brawler.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Pinned dependencies (torch comes from the Kaggle/Colab image)
!pip install -q -r tools/asset_gen/requirements-cuda.txt

In [ ]:
# Show the plan first (loads no model)
import shlex
args = ["--backend", "cuda", "--style", STYLE] + sum([["--only", o] for o in ONLY], []) + EXTRA_ARGS
ARGS = " ".join(shlex.quote(a) for a in args)
!python tools/asset_gen/generate.py --dry-run $ARGS

In [ ]:
# Make the images (FLUX.1 [schnell], fp16; 4-bit + CPU offload on 16 GB cards)
!python tools/asset_gen/generate.py $ARGS

In [ ]:
# Check the provenance manifest (file, prompt, seed, model, license, sha256 for every image)
!python tools/asset_gen/generate.py --validate-manifest assets/generated/ASSET_MANIFEST.json

In [ ]:
# Zip everything for download
import shutil
zip_base = os.path.join(WORK, "plaza_brawler_assets")
path = shutil.make_archive(zip_base, "zip", "assets/generated")
print("Zip:", path, f"{os.path.getsize(path)/2**20:.1f} MB")
if ON_KAGGLE:
    from IPython.display import FileLink, display
    os.chdir(WORK); display(FileLink(os.path.basename(path)))
    print("Or: right panel → Output → plaza_brawler_assets.zip → Download")
elif ON_COLAB:
    from google.colab import files
    files.download(path)

In [ ]:
# Quick look at what came out
from IPython.display import display
from PIL import Image
import glob
for p in sorted(glob.glob(os.path.join(REPO, "assets/generated/sheets/*.png")))[:2] + sorted(glob.glob(os.path.join(REPO, "assets/generated/stage/*.png")))[:1]:
    im = Image.open(p); im.thumbnail((1200, 1200)); print(p); display(im)